# ABS Residential Property Price Index (long-run, spliced)

House prices vs wages (AWOTE), indexed to 100 at the common start.

The house-price line splices three ABS measures into one continuous series:
the current all-dwellings mean price (6432.0), the discontinued Residential
Property Price Index for the eight capital cities (6416.0, to 2021Q4), and the
long-run established-house price index (6416.0 Table 8, back to 1986).


## Python set-up

In [1]:
# system imports
from functools import cache

In [2]:
# analytic imports
import pandas as pd
import readabs as ra
from abs_prices import get_cpi, get_wage_index
from readabs import metacol as mc

In [3]:
# local imports
import mgplot as mg
from mgplot import line_plot_finalise, multi_start

# pandas display settings
pd.options.display.max_rows = 999999
pd.options.display.max_columns = 999

# display charts within this notebook
SHOW = False
FILE_TYPE = "png"

In [4]:
# Constants
# Discontinued 6416.0 final release (Dec 2021) landing page - readabs url override.
RPPI_URL = (
    "https://www.abs.gov.au/statistics/economy/price-indexes-and-inflation/"
    "residential-property-price-indexes-eight-capital-cities/dec-2021"
)
source = "ABS: 6416.0, 6432.0, 6302.0"
WEEKS_PER_YEAR = 365.24 / 7  # annualise weekly AWOTE

# Output chart directory (fresh for this notebook).
mg.set_chart_dir("./CHARTS/6416.0 - Residential Property Price Index/")
mg.clear_chart_dir()

## Fetch and splice the house-price index

In [5]:
def get_house_price_index() -> tuple[pd.Series, pd.DataFrame]:
    """Splice a continuous long-run house-price index (highest priority first).

    1. 6432.0 mean dwelling value (all dwellings, 2011Q3 to current).
    2. 6416.0 Table 1 RPPI, weighted average of eight capitals (2003Q3-2021Q4).
    3. 6416.0 Table 8 established-house index (houses only, 1986Q2-2005Q2).

    rebase=True multiplicatively chains each lower-priority index onto the
    running level at the overlap, so all three measures join smoothly.
    """
    # 1. current all-dwellings mean price
    d, m = ra.read_abs_cat("6432.0")
    _t, sid, _u = ra.find_abs_id(
        m, {"Mean price of residential dwellings ;  Australia ;": mc.did}
    )
    mean_val = d[_t][sid].dropna() * 1_000  # $'000 -> $
    mean_val.index = mean_val.index.asfreq("Q-DEC")

    # 2. and 3. discontinued RPPI + long-run established-house index
    d6, _m6 = ra.read_abs_cat("6416.0", url=RPPI_URL)
    rppi = d6["641601"]["A83728455L"].dropna()       # RPPI, 8 capitals
    rppi.index = rppi.index.asfreq("Q-DEC")
    est = d6["641608"]["A2333613R"].dropna()         # established homes, 8 capitals
    est.index = est.index.asfreq("Q-DEC")

    spliced, report = ra.splice(
        [mean_val, rppi, est], rebase=True, name="House price index"
    )
    return spliced, report

## Chart: house prices vs wages (AWOTE)

In [8]:
def plot_hpi_vs_awote() -> pd.DataFrame:
    """House-price index vs AWOTE wage index, both 100 at the common start."""
    hpi, report = get_house_price_index()
    awote, _awote_units, awote_stype = get_wage_index("AWOTE")

    base = max(hpi.index.min(), awote.index.min())   # 1994Q4 (AWOTE start)
    df = pd.DataFrame(
        {
            "House price index": hpi / hpi.loc[base] * 100,
            "AWOTE (wages)": awote / awote.loc[base] * 100,
        }
    )
    df = df[df.index >= base]

    multi_start(
        df,
        function=line_plot_finalise,
        starts=[0, -41],
        title=f"House prices vs wages (AWOTE): {base} = 100",
        ylabel=f"Index ({base} = 100)",
        rfooter=source,
        lfooter=(
            "Australia. House-price index spliced from ABS 6432.0 and 6416.0. "
            f"AWOTE: {awote_stype.lower()}. "
        ),
        lheader="Pre-2003 house price uses the established-house index (pre-2005 methodology). ",
        width=2,
        annotate=True,
        rounding=0,
        color=["darkblue", "darkorange"],
        axhline={"y": 100, "color": "grey", "linestyle": "--", "lw": 0.75},
        show=SHOW,
        file_type=FILE_TYPE,
    )
    return report


splice_report = plot_hpi_vs_awote()
splice_report

,segment,name,freq_in,method,overlap_n,window_start,window_end,factor,fills_from
0,1,A83728455L,Q-DEC,window,42,2011Q3,2021Q4,4772.738784,2003Q3
1,2,A2333613R,Q-DEC,window,8,2003Q3,2005Q2,1367.004291,1986Q2


## Chart: house-price to income (AWOTE) ratio

In [9]:
def plot_price_to_income() -> None:
    """Mean dwelling price as a multiple of annual AWOTE (years of earnings)."""
    hpi, _report = get_house_price_index()
    awote, _awote_units, _awote_stype = get_wage_index("AWOTE")
    ratio = (hpi / (awote * WEEKS_PER_YEAR)).dropna()

    multi_start(
        ratio,
        function=line_plot_finalise,
        starts=[0, -41],
        title="Mean dwelling price to income (AWOTE) ratio",
        ylabel="Years of annual earnings",
        rfooter=f"Extrapolated from {source}",
        lfooter=(
            "Australia. Mean dwelling price / annual AWOTE "
            "(full-time adult ordinary earnings). "
        ),
        lheader="Pre-2003 house price uses the established-house index (pre-2005 methodology). ",
        width=2,
        annotate=True,
        rounding=1,
        color=["darkred"],
        show=SHOW,
        file_type=FILE_TYPE,
    )


plot_price_to_income()

## Chart: house prices vs CPI (back to 1986)

In [10]:
def plot_resi_vs_cpi() -> None:
    """House-price index vs CPI, both 100 at the common start (1986Q4)."""
    hpi, _report = get_house_price_index()
    cpi, _cpi_units, cpi_stype = get_cpi("headline_sa")

    base = max(hpi.index.min(), cpi.index.min())   # 1986Q4 (CPI start)
    df = pd.DataFrame(
        {
            "House price index": hpi / hpi.loc[base] * 100,
            "CPI (All groups, SA)": cpi / cpi.loc[base] * 100,
        }
    )
    df = df[df.index >= base]

    multi_start(
        df,
        function=line_plot_finalise,
        starts=[0, -41],
        title=f"House prices vs CPI: {base} = 100",
        ylabel=f"Index ({base} = 100)",
        rfooter="ABS: 6416.0, 6432.0, 6401.0",
        lfooter=(
            "Australia. House-price index spliced from ABS 6432.0 and 6416.0. "
            f"CPI: {cpi_stype.lower()}. "
        ),
        lheader="Pre-2003 house price uses the established-house index (pre-2005 methodology). ",
        width=2,
        annotate=True,
        rounding=0,
        color=["darkblue", "green"],
        axhline={"y": 100, "color": "grey", "linestyle": "--", "lw": 0.75},
        show=SHOW,
        file_type=FILE_TYPE,
    )


plot_resi_vs_cpi()

## Watermark

In [11]:
# watermark
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-06-12 10:14:20

Python implementation: CPython
Python version       : 3.14.2
IPython version      : 9.14.0

conda environment: n/a

Compiler    : Clang 21.1.4 
OS          : Darwin
Release     : 25.5.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot : 0.2.26
pandas : 3.0.3
readabs: 0.2.2

Watermark: 2.6.0

